In [23]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import plotly.express as px

In [24]:
df = pd.read_csv('data/diabetes_data.csv')
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 778 entries, 0 to 777
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               778 non-null    int64  
 1   Glucose                   778 non-null    int64  
 2   BloodPressure             778 non-null    int64  
 3   SkinThickness             778 non-null    int64  
 4   Insulin                   778 non-null    int64  
 5   BMI                       778 non-null    float64
 6   DiabetesPedigreeFunction  778 non-null    float64
 7   Age                       778 non-null    int64  
 8   Outcome                   778 non-null    int64  
 9   Gender                    778 non-null    object 
dtypes: float64(2), int64(7), object(1)
memory usage: 60.9+ KB


None

 все повторяющиеся строки в данных и удаление

In [25]:
# Проверка на наличие дубликатов
print(f"Количество дубликатов в данных: {df.duplicated().sum()}")

# Удаление дубликатов
df = df.drop_duplicates()

# Проверка количества оставшихся строк
print(f"Количество записей после удаления дубликатов: {df.shape[0]}")


Количество дубликатов в данных: 10
Количество записей после удаления дубликатов: 768


все неинформативные признаки в данных и избавемся от них

In [26]:
# Инициализация списка для неинформативных признаков
uninformative_features = []

# Проверка каждого признака на порог информативности
for column in df.columns:
    # Доля уникальных значений
    unique_fraction = df[column].nunique() / len(df)
    # Доля повторяющихся значений
    most_common_fraction = df[column].value_counts(normalize=True).iloc[0]
    
    # Если более 95% уникальны или повторяются, добавляем в список
    if unique_fraction > 0.95 or most_common_fraction > 0.95:
        uninformative_features.append(column)

# Удаление неинформативных признаков
df = df.drop(columns=uninformative_features)

# Вывод найденных неинформативных признаков
print(f"Неинформативные признаки: {uninformative_features}")
print(f"Количество оставшихся признаков: {df.shape[1]}")


Неинформативные признаки: ['Gender']
Количество оставшихся признаков: 9


поиск пустой информации

In [27]:
import numpy as np

# Список столбцов, в которых нужно заменить 0 на np.nan
columns_to_replace = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# Замена 0 на np.nan
df[columns_to_replace] = df[columns_to_replace].replace(0, np.nan)

# Вычисление доли пропусков в столбце 'Insulin'
missing_fraction = df["Insulin"].isnull().mean()

# Вывод результата
print(f"Доля пропусков в столбце 'Insulin': {missing_fraction:.2f}")


Доля пропусков в столбце 'Insulin': 0.49


удаление

In [28]:
# Определяем порог для удаления признаков
threshold = 0.3

# Удаляем признаки с долей пропусков > 30%
df = df.loc[:, df.isnull().mean() <= threshold]

# Вывод оставшихся признаков
print(f"Количество оставшихся признаков: {df.shape[1]}")

# Удаление строк с более чем двумя пропусками
df = df[df.isnull().sum(axis=1) <= 2]

# Вывод количества оставшихся записей
print(f"Количество оставшихся записей в таблице: {df.shape[0]}")



Количество оставшихся признаков: 8
Количество оставшихся записей в таблице: 761


замена пропусков

In [29]:
# Замена пропусков на медиану
df = df.fillna(df.median(numeric_only=True))

# Вычисление среднего значения в столбце SkinThickness
mean_skin_thickness = df["SkinThickness"].mean()

# Вывод результата
print(f"Среднее значение в столбце SkinThickness: {mean_skin_thickness:.1f}")


Среднее значение в столбце SkinThickness: 29.1


классический метод межквартильного размаха в признаке SkinThickness

In [30]:
# Вычисляем первый и третий квартиль
Q1 = df["SkinThickness"].quantile(0.25)
Q3 = df["SkinThickness"].quantile(0.75)

# Расчёт межквартильного размаха (IQR)
IQR = Q3 - Q1

# Определение границ выбросов
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Нахождение выбросов
outliers = df[(df["SkinThickness"] < lower_bound) | (df["SkinThickness"] > upper_bound)]

# Количество выбросов
print(f"Количество выбросов в 'SkinThickness': {outliers.shape[0]}")


Количество выбросов в 'SkinThickness': 87


классический метод z-отклонения в признаке SkinThickness?

In [31]:
import numpy as np

# Вычисление z-отклонений
mean = df["SkinThickness"].mean()
std = df["SkinThickness"].std()
z_scores = (df["SkinThickness"] - mean) / std

# Определение выбросов (порог ±3 стандартных отклонения)
outliers = df[np.abs(z_scores) > 3]

# Вывод количества выбросов
print(f"Количество выбросов в 'SkinThickness' по методу z-отклонений: {outliers.shape[0]}")


Количество выбросов в 'SkinThickness' по методу z-отклонений: 4


DiabetesPedigreeFunction и его логорифмирование

In [33]:
# Первый и третий квартиль
Q1 = df["DiabetesPedigreeFunction"].quantile(0.25)
Q3 = df["DiabetesPedigreeFunction"].quantile(0.75)

# Межквартильный размах
IQR = Q3 - Q1

# Границы выбросов
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Найти выбросы
outliers_original = df[(df["DiabetesPedigreeFunction"] < lower_bound) | (df["DiabetesPedigreeFunction"] > upper_bound)]
count_original = outliers_original.shape[0]
print(f"Количество выбросов в оригинальном масштабе: {count_original}")


# Логарифмирование признака
log_diabetes_pedigree = np.log(df["DiabetesPedigreeFunction"])

# Новый IQR
Q1_log = log_diabetes_pedigree.quantile(0.25)
Q3_log = log_diabetes_pedigree.quantile(0.75)
IQR_log = Q3_log - Q1_log

# Границы выбросов
lower_bound_log = Q1_log - 1.5 * IQR_log
upper_bound_log = Q3_log + 1.5 * IQR_log

# Найти выбросы
outliers_log = df[(log_diabetes_pedigree < lower_bound_log) | (log_diabetes_pedigree > upper_bound_log)]
count_log = outliers_log.shape[0]
print(f"Количество выбросов в логарифмическом масштабе: {count_log}")


Количество выбросов в оригинальном масштабе: 29
Количество выбросов в логарифмическом масштабе: 0
